# Codex: Evaluating Large Language Models Trained on Code

## Learning Objectives
1. Understand decoder-only architecture for code generation
2. Implement few-shot code generation with in-context learning
3. Implement pass@k evaluation metric and statistical analysis
4. Analyze failure modes and compare with CodeT5

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from typing import List, Tuple, Dict
import subprocess
import signal
import time
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
np.random.seed(42)
torch.manual_seed(42)

## Level 1: Basic Few-Shot Code Generation

In [ ]:
def build_few_shot_prompt(problem_description: str, examples: List[str], temperature: float = 0.8) -> str:
    '''Build few-shot prompt for Codex-style code generation.
    
    Few-shot learning allows Codex to solve new problems by seeing example solutions.
    The prompt structure is critical for quality outputs.
    '''
    prompt = '# Few-shot examples of code solutions:\n\n'
    
    for i, example in enumerate(examples, 1):
        prompt += f'# Example {i}:\n'
        prompt += example
        prompt += '\n\n'
    
    prompt += '# New Problem:\n'
    prompt += problem_description
    prompt += '\n\n# Solution:\n'
    
    return prompt

# Test prompt construction
problem = '''def fibonacci(n):
    """Return the nth Fibonacci number (0-indexed)."""
'''

examples = [
    '''def factorial(n):
    """Return n factorial."""
    if n <= 1:
        return 1
    return n * factorial(n - 1)
''',
    '''def sum_to_n(n):
    """Return sum of integers from 1 to n."""
    total = 0
    for i in range(1, n + 1):
        total += i
    return total
''',
]

prompt = build_few_shot_prompt(problem, examples, temperature=0.8)
print('Few-shot Prompt Generated:')
print('=' * 60)
print(prompt)
print('=' * 60)
print(f'\nPrompt length: {len(prompt)} characters')
print(f'Temperature (diversity): 0.8')
print('✓ Few-shot setup complete')

In [ ]:
print(f'\nPrompt analysis:')
print(f'  Examples: {len(examples)}')
print(f'  Problem description length: {len(problem)} chars')
print(f'  Total prompt length: {len(prompt)} chars')
print(f'  Ready for Codex API call')

## Level 2: Execution-Based Code Evaluation with Pass@k

In [ ]:
def execute_code_safely(code: str, test_cases: List[Tuple[str, str]], timeout_sec: int = 5, function_name: str = None) -> Dict[str, any]:
    '''Execute generated code and check correctness against test cases.
    
    This is the critical function for evaluating code generation models.
    It safely executes untrusted code and categorizes any failures.
    
    Args:
        code: Generated Python code
        test_cases: List of (input_args_str, expected_output_str) tuples
        timeout_sec: Max execution time per test
        function_name: Name of function to call (auto-detect if None)
    
    Returns:
        Dict with: passed (bool), num_passed (int), num_total (int), failure_mode (str)
    '''
    
    # Step 1: Check syntax
    try:
        compile(code, '<string>', 'exec')
    except SyntaxError as e:
        return {
            'passed': False,
            'num_passed': 0,
            'num_total': len(test_cases),
            'failure_mode': 'syntax_error',
            'error_message': f'SyntaxError: {str(e)}'
        }
    
    num_passed = 0
    failure_details = []
    detected_failure_mode = None
    
    # Step 2: Execute code and test
    for test_idx, (test_input, expected_output) in enumerate(test_cases):
        try:
            # Create isolated execution environment
            local_vars = {}
            global_vars = {}
            
            # Execute the code
            exec(code, global_vars, local_vars)
            
            # Find the function to call
            func = None
            if function_name:
                func = local_vars.get(function_name)
            else:
                # Auto-detect first callable
                for key, val in local_vars.items():
                    if callable(val) and not key.startswith('_'):
                        func = val
                        break
            
            if func is None:
                if not detected_failure_mode:
                    detected_failure_mode = 'runtime_error'
                failure_details.append({
                    'test': test_idx,
                    'error': 'No function definition found'
                })
                continue
            
            # Parse input arguments
            try:
                args = eval(f'[{test_input}]')
            except:
                args = [test_input]
            
            # Execute function
            result = func(*args)
            
            # Check output
            if str(result) == expected_output:
                num_passed += 1
            else:
                if not detected_failure_mode:
                    detected_failure_mode = 'logic_error'
                failure_details.append({
                    'test': test_idx,
                    'expected': expected_output,
                    'got': str(result)
                })
        
        except TimeoutError:
            if not detected_failure_mode:
                detected_failure_mode = 'timeout'
            failure_details.append({
                'test': test_idx,
                'error': f'Timeout after {timeout_sec}s'
            })
        except Exception as e:
            if not detected_failure_mode:
                detected_failure_mode = 'runtime_error'
            failure_details.append({
                'test': test_idx,
                'error': f'{type(e).__name__}: {str(e)}'
            })
    
    return {
        'passed': num_passed == len(test_cases),
        'num_passed': num_passed,
        'num_total': len(test_cases),
        'failure_mode': detected_failure_mode,
        'error_details': failure_details
    }

# Test on simple addition function
test_code = 'def add(a, b):\n    return a + b'
test_cases = [('1, 2', '3'), ('5, 5', '10'), ('0, 0', '0'), ('-5, 5', '0')]

print('Testing code execution:')
result = execute_code_safely(test_code, test_cases)
print(f'  Passed: {result["passed"]}')
print(f'  Score: {result["num_passed"]}/{result["num_total"]}')
print(f'  Failure mode: {result["failure_mode"]}')
print('✓ Execution testing complete')

In [ ]:
# Test multiple samples for pass@k evaluation
code_samples = [
    ('Correct', 'def add(a, b):\n    return a + b'),
    ('Correct variant', 'def add(x, y):\n    result = x + y\n    return result'),
    ('Wrong logic', 'def add(a, b):\n    return a * b'),
    ('Syntax error', 'def add(a, b)\n    return a + b'),
]

print('\nPass@k Evaluation:')
print('=' * 70)

results = []
for name, code in code_samples:
    result = execute_code_safely(code, test_cases)
    results.append(result)
    status = 'PASS' if result['passed'] else 'FAIL'
    print(f'  {name:20} {status:5} ({result["num_passed"]}/{result["num_total"]})')

# Compute pass@k metrics
num_correct = sum(1 for r in results if r['passed'])
print(f'\nPass@k Summary:')
print(f'  Pass@1 (1st sample): {results[0]["passed"]}')
print(f'  Pass@2 (at least 1 of 2): {any(r["passed"] for r in results[:2])}')
print(f'  Pass@4 (at least 1 of 4): {num_correct > 0}')
print(f'  Correct samples: {num_correct}/{len(code_samples)}')
print(f'\n✓ Pass@k evaluation complete')

## Real-World Example 1: Function Implementation from Docstring

In [ ]:
def evaluate_multiple_solutions(problem_name: str, test_cases: List[Tuple[str, str]], candidate_codes: List[Tuple[str, str]]) -> Dict:
    '''Evaluate multiple code generation attempts for a problem.
    
    This simulates Codex generating k samples and evaluating how many solve the problem.
    '''
    
    results = {
        'problem': problem_name,
        'num_samples': len(candidate_codes),
        'num_correct': 0,
        'evaluations': [],
        'correct_indices': []
    }
    
    for i, (name, code) in enumerate(candidate_codes):
        eval_result = execute_code_safely(code, test_cases)
        eval_result['name'] = name
        results['evaluations'].append(eval_result)
        
        if eval_result['passed']:
            results['num_correct'] += 1
            results['correct_indices'].append(i)
    
    return results

# Test with factorial implementation
factorial_tests = [('0', '1'), ('1', '1'), ('5', '120'), ('10', '3628800'), ('3', '6')]

factorial_candidates = [
    ('Recursive', 'def factorial(n):\n    if n <= 1:\n        return 1\n    return n * factorial(n - 1)'),
    ('Iterative', 'def factorial(n):\n    result = 1\n    for i in range(1, n + 1):\n        result *= i\n    return result'),
    ('With guard', 'def factorial(n):\n    if n < 0:\n        return None\n    if n <= 1:\n        return 1\n    return n * factorial(n - 1)'),
    ('One-liner', 'def factorial(n):\n    return 1 if n <= 1 else n * factorial(n - 1)'),
    ('Math module', 'import math\ndef factorial(n):\n    return math.factorial(n)'),
]

eval_summary = evaluate_multiple_solutions('factorial', factorial_tests, factorial_candidates)

print('Function Generation Evaluation Results:')
print('=' * 70)
print(f'Problem: {eval_summary["problem"]}')
print(f'Total samples: {eval_summary["num_samples"]}')
print(f'Correct solutions: {eval_summary["num_correct"]}')
print(f'\nDetailed Results:')
for i, eval_res in enumerate(eval_summary['evaluations']):
    status = '✓' if eval_res['passed'] else '✗'
    print(f'  Sample {i} ({eval_res["name"]:15}): {status} ({eval_res["num_passed"]}/{eval_res["num_total"]}) - {eval_res["failure_mode"]}')

print(f'\nPass@k Metrics:')
print(f'  Pass@1: {eval_summary["evaluations"][0]["passed"]}')
print(f'  Pass@5: {eval_summary["num_correct"] > 0}')
print(f'  Correct sample indices: {eval_summary["correct_indices"]}')
print('✓ Function evaluation complete')

## Real-World Example 2: Bug Fixing Evaluation

In [ ]:
class CodeGenerationEvaluator:
    '''Comprehensive evaluator for code generation models.
    
    Supports multiple use cases: function generation, bug fixing, optimization.
    '''
    
    def __init__(self):
        self.evaluation_history = []
    
    def find_first_working_sample(self, samples: List[Tuple[str, str]], test_cases: List) -> Tuple[int, str, bool]:
        '''Find first sample that passes all tests (pass@k metric).
        
        Returns:
            (sample_index, sample_name, found)
        '''
        for i, (name, code) in enumerate(samples):
            if execute_code_safely(code, test_cases)['passed']:
                return i, name, True
        return -1, None, False
    
    def evaluate_on_problem(self, test_cases: List, samples: List[Tuple[str, str]]) -> Dict:
        '''Evaluate all samples on a problem.
        
        Returns:
            Dict with pass@1, pass@k, correct indices
        '''
        results = []
        for name, code in samples:
            eval_result = execute_code_safely(code, test_cases)
            results.append({'name': name, 'passed': eval_result['passed']})
        
        correct_count = sum(1 for r in results if r['passed'])
        return {
            'pass@1': results[0]['passed'] if results else False,
            f'pass@{len(results)}': correct_count > 0,
            'correct_count': correct_count,
            'results': results
        }

# Test bug fixing scenario
evaluator = CodeGenerationEvaluator()

# Prime number checking function
prime_tests = [('2', 'True'), ('1', 'False'), ('4', 'False'), ('17', 'True'), ('100', 'False'), ('97', 'True')]

prime_attempts = [
    ('Buggy original', 'def is_prime(n):\n    if n < 2:\n        return True\n    for i in range(2, n):\n        if n % i == 0:\n            return False\n    return True'),
    ('Fixed', 'def is_prime(n):\n    if n < 2:\n        return False\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0:\n            return False\n    return True'),
    ('Alternative fix', 'def is_prime(n):\n    if n <= 1:\n        return False\n    if n == 2:\n        return True\n    for i in range(2, n):\n        if n % i == 0:\n            return False\n    return True'),
]

first_fix_idx, first_fix_name, found = evaluator.find_first_working_sample(prime_attempts, prime_tests)

print('Bug Fix Evaluation (is_prime function):')
print('=' * 70)
print(f'Found working fix: {found}')
if found:
    print(f'First working at index: {first_fix_idx} ({first_fix_name})')
    
eval_results = evaluator.evaluate_on_problem(prime_tests, prime_attempts)
print(f'\nResults:')
for res in eval_results['results']:
    status = '✓' if res['passed'] else '✗'
    print(f'  {status} {res["name"]}')

print(f'\nPass@k Metrics:')
print(f'  Pass@1: {eval_results["pass@1"]}')
print(f'  Pass@3: {eval_results[f"pass@3"]}')
print(f'  Correct samples: {eval_results["correct_count"]}')
print('✓ Bug fix evaluation complete')

## Key Takeaways

In [ ]:
print('Key Insights:')
print('1. Few-shot learning: Examples in prompt guide generation')
print('2. Pass@k metric: Probability at least 1 of k samples solves problem')
print('3. Failure modes: Syntax, logic, timeout - each needs different fix')
print('4. Codex strength: Strong on generation, weak on understanding')
print('\n✓ Codex training complete')